# LeetCode #425: Word Squares

https://leetcode.com/problems/word-squares/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Try All Combinations)** | $O(n^L)$ | $O(L)$ |
| **Optimal: Backtracking + Trie/HashMap ★** | $O(n \cdot 26^L)$ worst case | $O(n \cdot L)$ |

---

## Understanding the Methods

### Brute Force (Try All Combinations)
Try every combination of L words (where L is word length) and check if they form a valid word square. Extremely slow.

### Optimal: Backtracking + Trie/HashMap ★
Build a prefix map from all words. Place words row by row using backtracking. For row i, the required prefix is formed by taking column i from all previously placed rows. Use the prefix map to quickly find candidate words, pruning invalid branches early.

**Why this is better than Brute Force:** The prefix map prunes the search space dramatically by only considering words that match the required prefix at each step.

**Constraints:**
* 1 <= words.length <= 1000
* 1 <= words[i].length <= 4
* All words[i] have the same length
* words[i] consists of only lowercase English letters
* All words[i] are unique

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<string>> WordSquares(string[] words) {
        int len = words[0].Length;
        var prefixMap = new Dictionary<string, List<string>>();
        foreach (string w in words) {
            for (int i = 0; i <= w.Length; i++) {
                string p = w.Substring(0, i);
                if (!prefixMap.ContainsKey(p)) prefixMap[p] = new List<string>();
                prefixMap[p].Add(w);
            }
        }
        var result = new List<IList<string>>();
        var square = new List<string>();
        void Backtrack(int row) {
            if (row == len) { result.Add(new List<string>(square)); return; }
            var sb = new System.Text.StringBuilder();
            for (int i = 0; i < row; i++) sb.Append(square[i][row]);
            string prefix = sb.ToString();
            if (!prefixMap.ContainsKey(prefix)) return;
            foreach (string w in prefixMap[prefix]) {
                square.Add(w);
                Backtrack(row + 1);
                square.RemoveAt(square.Count - 1);
            }
        }
        Backtrack(0);
        return result;
    }
}

### Python

In [ ]:
from collections import defaultdict

class Solution:
    def wordSquares(self, words: list[str]) -> list[list[str]]:
        n = len(words[0])
        prefix_map = defaultdict(list)
        for w in words:
            for i in range(n + 1):
                prefix_map[w[:i]].append(w)

        result = []
        def backtrack(square):
            row = len(square)
            if row == n:
                result.append(list(square))
                return
            prefix = ''.join(square[i][row] for i in range(row))
            for w in prefix_map[prefix]:
                square.append(w)
                backtrack(square)
                square.pop()

        backtrack([])
        return result

### Go

In [ ]:
func wordSquares(words []string) [][]string {
    n := len(words[0])
    prefixMap := map[string][]string{}
    for _, w := range words {
        for i := 0; i <= n; i++ {
            p := w[:i]
            prefixMap[p] = append(prefixMap[p], w)
        }
    }
    var result [][]string
    var backtrack func([]string)
    backtrack = func(square []string) {
        row := len(square)
        if row == n {
            cp := make([]string, n)
            copy(cp, square)
            result = append(result, cp)
            return
        }
        prefix := make([]byte, row)
        for i := 0; i < row; i++ {
            prefix[i] = square[i][row]
        }
        for _, w := range prefixMap[string(prefix)] {
            backtrack(append(square, w))
            square = square[:len(square)-1]
        }
    }
    backtrack(nil)
    return result
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn word_squares(words: Vec<String>) -> Vec<Vec<String>> {
        let n = words[0].len();
        let mut prefix_map: HashMap<String, Vec<usize>> = HashMap::new();
        for (idx, w) in words.iter().enumerate() {
            for i in 0..=n {
                prefix_map.entry(w[..i].to_string()).or_default().push(idx);
            }
        }
        let mut result = Vec::new();
        let mut square: Vec<usize> = Vec::new();
        fn backtrack(
            words: &[String], pm: &HashMap<String, Vec<usize>>,
            sq: &mut Vec<usize>, n: usize, res: &mut Vec<Vec<String>>,
        ) {
            let row = sq.len();
            if row == n {
                res.push(sq.iter().map(|&i| words[i].clone()).collect());
                return;
            }
            let prefix: String = (0..row).map(|i| words[sq[i]].as_bytes()[row] as char).collect();
            if let Some(candidates) = pm.get(&prefix) {
                for &c in candidates {
                    sq.push(c);
                    backtrack(words, pm, sq, n, res);
                    sq.pop();
                }
            }
        }
        backtrack(&words, &prefix_map, &mut square, n, &mut result);
        result
    }
}

## Example Scenarios

### Scenario 1: Standard word square
**Input:** `words = ["area","lead","wall","lady","ball"]`  
One valid square: ["wall","area","lead","lady"]. Reading columns top-down gives the same words. **Output:** `[["wall","area","lead","lady"],["ball","area","lead","lady"]]`

### Scenario 2: Single-letter words
**Input:** `words = ["a","b"]`  
Each single letter forms a 1x1 word square. **Output:** `[["a"],["b"]]`

### Scenario 3: Two-letter words
**Input:** `words = ["ab","ac"]`  
For a 2x2 square starting with "ab", column 1 must start with "b" -- no match. Starting with "ac", column 1 starts with "c" -- no match. **Output:** `[]`

### Scenario 4: Palindrome words
**Input:** `words = ["abcd","bnrt","crm","dtye"]`  
If the words form a valid square, column reads equal row reads. **Output:** `[["abcd","bnrt","crm","dtye"]]` (if valid)

### Scenario 5: Multiple valid squares
**Input:** `words = ["abat","baba","atan","atal"]`  
Multiple arrangements may satisfy the word square property. **Output:** `[["abat","baba","atan","atal"]]`

![image](attachment:image.png)